# Week 5 Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

> **Goal:** Model agents as stateful graphs of nodes (steps) and edges (transitions),
> enabling self-correction loops, human-in-the-loop interrupts, and persistent state history.

**Stack:** `langgraph 1.2.9` · `langchain-core` · `langchain-anthropic` · `pydantic`  
**Model:** `claude-3-5-haiku-20241022` (or robust fallback for offline validation)

```bash
pip install langgraph langchain-core langchain-anthropic pydantic python-dotenv
```


## Setup & Environment Configuration


In [ ]:
import os, sys, json, time, warnings
from typing import TypedDict, List, Dict, Any, Optional
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv('project.env')

import langgraph
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

print(f'langgraph version: {importlib.metadata.version("langgraph") if "importlib" in sys.modules else "1.2.9"}')
print(f'API key set       : {bool(os.environ.get("ANTHROPIC_API_KEY"))}')


---
## Task 1: LangGraph Concepts & State Design

### LangGraph Core Building Blocks

1. **`StateGraph`**: The primary graph builder. Parameterized by a shared `State` structure.
2. **`Shared State`**: A `TypedDict` or `Pydantic` model representing the global state object passed to and mutated by nodes.
3. **`Nodes`**: Python functions `(state: State) -> dict` that execute processing steps and return partial state updates.
4. **`Edges`**: Direct connections defining static control flow from one node to the next.
5. **`Conditional Edges`**: Router functions `(state: State) -> str` that evaluate state to dynamically determine the next node.

### Workflow Scenario & Architecture Diagram

We are building an **Autonomous Technical Research & Revision Agent** that creates research reports, evaluates quality, automatically loops back for revision if quality is low, pauses for human approval, and publishes.

```
                  ┌─────────┐
                  │  START  │
                  └────┬────┘
                       │
                       ▼
                 ┌───────────┐
                 │  Planner  │
                 └─────┬─────┘
                       │
                       ▼
                 ┌───────────┐
                 │ Researcher│ (Extracts product/math/weather data)
                 └─────┬─────┘
                       │
                       ▼
                 ┌───────────┐
                 │ Generator │◄─────────────────┐
                 └─────┬─────┘                  │
                       │                        │
                       ▼                        │ Self-Correction Loop
                 ┌───────────┐                  │ (if score < 8.0
                 │  Critic / │                  │  AND rev_count < max)
                 │ Evaluator │                  │
                 └─────┬─────┘                  │
                       │                        │
             Conditional Edge ──────────────────┘
                       │
                       │ (if score >= 8.0 OR rev_count >= max)
                       ▼
              ┌─────────────────┐
              │ Human Approval  │ ──► [INTERRUPT BEFORE] Pauses for human
              └────────┬────────┘
                       │
                       │ (human approves: is_approved = True)
                       ▼
                 ┌───────────┐
                 │ Publisher │
                 └─────┬─────┘
                       │
                       ▼
                    ┌─────┐
                    │ END │
                    └─────┘
```


In [ ]:
# ── State Schema Design ────────────────────────────────────────────────────────
class ResearchState(TypedDict):
    topic: str               # Research query / topic
    target_audience: str     # Target audience (e.g., Executive, Technical)
    plan: List[str]          # Bulleted research outline
    research_notes: List[str]# Extracted ground-truth facts/data
    draft: str               # Current draft content
    critique: str            # Feedback from Critic node
    quality_score: float     # Score from 0.0 to 10.0
    revision_count: int      # Number of revisions attempted
    max_revisions: int       # Maximum allowed revisions
    is_approved: bool        # Human sign-off status
    final_output: str        # Formatted final published output

print('State Schema defined successfully with 11 state keys.')


---
## Task 2: Build & Test a Linear Graph

We first implement a 4-node linear workflow: `planner` -> `researcher` -> `generator` -> `formatter`.
We run it on a sample topic and print state updates after each node.


In [ ]:
# ── Define Node Functions for Linear Graph ────────────────────────────────────
def node_planner(state: ResearchState) -> Dict[str, Any]:
    print('  [Node: Planner] Generating research outline...')
    topic = state['topic']
    plan = [
        f'1. Define core concepts of {topic}',
        f'2. Analyze key technical advantages and market position',
        f'3. Provide actionable recommendations for {state["target_audience"]}'
    ]
    return {'plan': plan}

def node_researcher(state: ResearchState) -> Dict[str, Any]:
    print('  [Node: Researcher] Gathering data and facts...')
    topic = state['topic']
    notes = [
        f'Fact 1: {topic} combines hardware efficiency with intelligent software.',
        f'Fact 2: Benchmark rating averages 4.7/5 across industry reviews.',
        f'Fact 3: Price-to-performance ratio outperforms competitors by 25%.'
    ]
    return {'research_notes': notes}

def node_generator(state: ResearchState) -> Dict[str, Any]:
    rev = state.get('revision_count', 0)
    print(f'  [Node: Generator] Generating draft (Revision #{rev})...')
    topic = state['topic']
    plan_str = '\n'.join(state.get('plan', []))
    notes_str = '\n'.join(state.get('research_notes', []))
    
    draft = f"""# Technical Overview: {topic}
Audience: {state.get('target_audience', 'General')}

## Outline
{plan_str}

## Key Findings
{notes_str}
"""
    return {'draft': draft}

def node_formatter(state: ResearchState) -> Dict[str, Any]:
    print('  [Node: Formatter] Formatting final output...')
    formatted = f"""==================================================
EXECUTIVE REPORT: {state['topic'].upper()}
==================================================
{state['draft'].strip()}
=================================================="""
    return {'final_output': formatted}

# ── Assemble & Compile Linear Graph ───────────────────────────────────────────
linear_builder = StateGraph(ResearchState)
linear_builder.add_node('planner', node_planner)
linear_builder.add_node('researcher', node_researcher)
linear_builder.add_node('generator', node_generator)
linear_builder.add_node('formatter', node_formatter)

linear_builder.add_edge(START, 'planner')
linear_builder.add_edge('planner', 'researcher')
linear_builder.add_edge('researcher', 'generator')
linear_builder.add_edge('generator', 'formatter')
linear_builder.add_edge('formatter', END)

linear_app = linear_builder.compile()
print('Linear graph compiled successfully.')


In [ ]:
# ── Run Linear Graph & Inspect Intermediate State ─────────────────────────────
initial_state: ResearchState = {
    'topic': 'Apple MacBook Air M3',
    'target_audience': 'Executive Decision Makers',
    'plan': [],
    'research_notes': [],
    'draft': '',
    'critique': '',
    'quality_score': 0.0,
    'revision_count': 0,
    'max_revisions': 3,
    'is_approved': False,
    'final_output': ''
}

print('=== Executing Linear Graph ===')
output_state = linear_app.invoke(initial_state)
print('\n=== Final State Inspection ===')
for key, val in output_state.items():
    if isinstance(val, list):
        print(f'  {key:15s}: {len(val)} items')
    elif isinstance(val, str) and len(val) > 60:
        print(f'  {key:15s}: {val[:50]}... ({len(val)} chars)')
    else:
        print(f'  {key:15s}: {val}')


---
## Task 3: Add Conditional Edges & Cycles (Self-Correction Loop)

We introduce a **Critic Node** and a **Conditional Edge** `route_critique`.
- On Pass 1, the Critic evaluates the draft, gives feedback, and assigns a score (e.g. `6.5/10`).
- Because `6.5 < 8.0`, `route_critique` routes execution **back to `generator`**, incrementing `revision_count` to 1.
- On Pass 2, `generator` addresses the critique, and `critic` evaluates the revised draft at `8.5/10`.
- Because `8.5 >= 8.0`, `route_critique` routes forward to `human_approval` / `publisher`.
- `max_revisions = 3` acts as a hard safeguard preventing infinite loops.


In [ ]:
# ── Define Critic Node & Router Function ──────────────────────────────────────
def node_critic(state: ResearchState) -> Dict[str, Any]:
    rev = state.get('revision_count', 0)
    print(f'  [Node: Critic] Evaluating draft (Pass #{rev + 1})...')
    
    # Simulated evaluation: Pass 1 gets 6.5, Pass 2 gets 8.5
    if rev == 0:
        score = 6.5
        critique = 'Draft lacks specific benchmark numbers and pricing ROI comparison. Needs revision.'
    else:
        score = 8.5
        critique = 'Excellent draft with thorough benchmarks and executive ROI rationale. Approved.'
    
    print(f'    -> Score Assigned: {score}/10.0 | Critique: "{critique[:55]}..."')
    return {
        'quality_score': score,
        'critique': critique,
        'revision_count': rev + 1
    }

def route_critique(state: ResearchState) -> str:
    score = state.get('quality_score', 0.0)
    rev   = state.get('revision_count', 0)
    max_r = state.get('max_revisions', 3)
    
    print(f'  [Conditional Edge: route_critique] Check: Score={score}, RevCount={rev}, MaxRev={max_r}')
    
    if score >= 8.0:
        print('    -> Quality threshold met (>= 8.0)! Routing to human_approval.')
        return 'human_approval'
    elif rev >= max_r:
        print('    -> Max revisions reached! Routing to human_approval.')
        return 'human_approval'
    else:
        print('    -> Quality below threshold (< 8.0). Routing BACK to generator.')
        return 'generator'

def node_human_approval(state: ResearchState) -> Dict[str, Any]:
    print('  [Node: Human Approval] Awaiting human sign-off...')
    return {'is_approved': state.get('is_approved', False)}

def node_publisher(state: ResearchState) -> Dict[str, Any]:
    print('  [Node: Publisher] Publishing finalized report...')
    final_doc = f"""==================================================
PUBLISHED REPORT: {state['topic'].upper()}
Quality Score: {state['quality_score']}/10.0 | Approved: {state['is_approved']}
==================================================
{state['draft']}
Critique Summary: {state['critique']}
=================================================="""
    return {'final_output': final_doc}


In [ ]:
# ── Assemble & Compile Cyclical Graph ─────────────────────────────────────────
cyclical_builder = StateGraph(ResearchState)
cyclical_builder.add_node('planner', node_planner)
cyclical_builder.add_node('researcher', node_researcher)
cyclical_builder.add_node('generator', node_generator)
cyclical_builder.add_node('critic', node_critic)
cyclical_builder.add_node('human_approval', node_human_approval)
cyclical_builder.add_node('publisher', node_publisher)

cyclical_builder.add_edge(START, 'planner')
cyclical_builder.add_edge('planner', 'researcher')
cyclical_builder.add_edge('researcher', 'generator')
cyclical_builder.add_edge('generator', 'critic')

# Add Conditional Edge from critic
cyclical_builder.add_conditional_edges(
    'critic',
    route_critique,
    {
        'generator': 'generator',
        'human_approval': 'human_approval'
    }
)
cyclical_builder.add_edge('human_approval', 'publisher')
cyclical_builder.add_edge('publisher', END)

cyclical_app = cyclical_builder.compile()
print('Cyclical graph compiled successfully.')


In [ ]:
# ── Test Cyclical Graph (Self-Correction Demonstration) ───────────────────────
print('=== Executing Cyclical Self-Correction Graph ===')
res_cyclical = cyclical_app.invoke(initial_state)
print('\n=== Self-Correction Cycle Summary ===')
print(f'Total Revisions Attempted: {res_cyclical["revision_count"]}')
print(f'Final Quality Score     : {res_cyclical["quality_score"]}/10.0')
print(f'Final Critique          : {res_cyclical["critique"]}')


### Why Loop-backs are Natural in LangGraph vs Hard in AgentExecutor

> In `AgentExecutor`, the control loop is a fixed while-loop (`LLM -> Tool -> LLM -> Tool -> Answer`). Creating a multi-phase revision loop (Draft -> Evaluate -> Revise -> Re-evaluate) requires custom prompt engineering hacks, custom tool output parsers, or tricking the LLM to call fake tools. In contrast, `LangGraph` treats loops as first-class directional edges between graph nodes, governed by explicit conditional router functions and numeric state variables (`revision_count`, `quality_score`).


---
## Task 4: Human-in-the-Loop & Interrupts

We compile the graph with a `MemorySaver` checkpointer and set `interrupt_before=['publisher']`.
The agent executes through planning, research, drafting, critique, and human approval, then **pauses** automatically before executing the risky `publisher` step.
We inspect the paused state, simulate human approval (`app.update_state`), and resume execution.


In [ ]:
# ── Compile Graph with Checkpointer & Interrupt Point ─────────────────────────
memory_checkpointer = MemorySaver()
hitl_app = cyclical_builder.compile(
    checkpointer=memory_checkpointer,
    interrupt_before=['publisher']
)
print('HITL graph compiled with interrupt_before=["publisher"].')


In [ ]:
# ── Step 1: Run graph until interrupt point ───────────────────────────────────
thread_config = {'configurable': {'thread_id': 'session_hitl_101'}}

print('=== STEP 1: Running Graph until Interrupt Point ===')
hitl_app.invoke(initial_state, config=thread_config)

# Check current state at the pause point
snapshot = hitl_app.get_state(thread_config)
print('\n=== GRAPH PAUSED ===')
print(f'Next node to execute : {snapshot.next}')
print(f'Current is_approved   : {snapshot.values.get("is_approved")}')
print(f'Current draft preview : {snapshot.values.get("draft")[:60]}...')


In [ ]:
# ── Step 2: Human Inspection & Approval State Update ──────────────────────────
print('=== STEP 2: Human Review & Approval ===')
print('Human reviewer inspects draft and approves deployment.')

# Update state to mark is_approved = True
hitl_app.update_state(
    thread_config,
    {'is_approved': True}
)

post_update_snapshot = hitl_app.get_state(thread_config)
print(f'Updated is_approved status: {post_update_snapshot.values.get("is_approved")}')


In [ ]:
# ── Step 3: Resume Execution ───────────────────────────────────────────────────
print('=== STEP 3: Resuming Graph Execution ===')
final_res = hitl_app.invoke(None, config=thread_config)
print('\n=== FINAL PUBLISHED OUTPUT ===')
print(final_res['final_output'])


### When Should a Product Require Human-in-the-Loop vs. Full Autonomy?

| Dimension | Require Human-in-the-Loop | Permit Full Autonomy |
|---|---|---|
| **Impact / Consequence** | High risk (financial trades, legal contracts, sending emails) | Low risk (read-only search, internal summaries) |
| **Reversibility** | Irreversible side-effects (API POST, database delete) | Reversible / cached operations |
| **Accuracy Need** | Zero-tolerance for hallucination (medical/regulatory) | Exploratory brainstorming or draft generation |
| **Compliance** | Mandated human oversight | Non-regulated tasks |


---
## Task 5: Persistence & State History Debugging

We use `MemorySaver` to demonstrate:
1. **Multi-Session Persistence**: Resuming workflow sessions using `thread_id`.
2. **Time-Travel Debugging**: Traversing historical state snapshots via `get_state_history()`.
3. **Framework Comparison**: `AgentExecutor` vs `LangGraph` decision matrix.


In [ ]:
# ── Time-Travel & State History Inspection ────────────────────────────────────
print('=== State History / Time-Travel Debugging ===')
history = list(hitl_app.get_state_history(thread_config))
print(f'Total historical state checkpoints recorded: {len(history)}\n')

for idx, state_snapshot in enumerate(reversed(history)):
    next_node = state_snapshot.next
    rev_cnt   = state_snapshot.values.get('revision_count', 0)
    score     = state_snapshot.values.get('quality_score', 0.0)
    print(f'Checkpoint #{idx+1:02d} | Next Node: {str(next_node):20s} | Score: {score}/10.0 | Revisions: {rev_cnt}')


### Framework Comparison: LangChain `AgentExecutor` vs. `LangGraph`

| Feature / Metric | LangChain AgentExecutor | LangGraph |
|---|---|---|
| **Control Flow** | Monolithic while-loop around tool calling | Expressive Graph (Nodes + Edges + Cycles) |
| **State Management** | Implicit string message list | Explicit, typed shared State dictionary |
| **Multi-Step Workflows** | Hard to control / fragile prompts | Native graph node decomposition |
| **Self-Correction Loops** | Clunky / unnatural | Native via conditional edges & max counter |
| **Human-in-the-Loop** | Fragile tool-level hacks | Native `interrupt_before` / `interrupt_after` |
| **Time-Travel Debugging** | Not available | Native via `get_state_history()` |
| **When to Use** | Simple single-turn tool Q&A | Enterprise workflows, multi-agent systems, complex stateful pipelines |

**Summary Recommendation:** Use `AgentExecutor` for simple, quick tool-assisted chat assistants. Switch to `LangGraph` for production applications requiring custom business logic, multi-stage review pipelines, human approval gates, state persistence, or multi-agent orchestration.
